# DeepSeek OCR V2 — Extract Text from PDFs on Kaggle

This notebook extracts text from PDF files using the **DeepSeek-OCR-2** model and saves results as Markdown.

**Workflow:**
1. Clone the project repo from GitHub
2. Download PDF data from Google Drive
3. Install dependencies
4. Run the OCR pipeline
5. **Save Version** on Kaggle to preserve all outputs

> **Tip:** Use Kaggle's **Save Version → Save & Run All** to run the entire pipeline in one shot.
> Output Markdown files will appear directly in the **Output** tab after the run completes.

## 1. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/hoangtung386/DeepSeek-OCR-V2-for-PDFs-on-Kaggle.git"
PROJECT_DIR = "/kaggle/working/DeepSeek-OCR-V2-for-PDFs-on-Kaggle"

# Output goes to /kaggle/working/output/ so it shows up in Kaggle's Output tab
DATA_DIR = os.path.join(PROJECT_DIR, "data")
OUTPUT_DIR = "/kaggle/working/output"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Data directory:    {DATA_DIR}")
print(f"Output directory:  {OUTPUT_DIR}")

## 2. Download PDF Data from Google Drive

In [ ]:
!pip install -q gdown

import gdown

GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1PqauuwAzZsSCk7TdyRYkRSDbwCBxb0Aj"

os.makedirs(DATA_DIR, exist_ok=True)
gdown.download_folder(GDRIVE_FOLDER_URL, output=DATA_DIR, quiet=False)

pdf_count = len([f for f in os.listdir(DATA_DIR) if f.lower().endswith(".pdf")])
print(f"\nDownloaded {pdf_count} PDF file(s) to {DATA_DIR}")

## 3. Install Dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq poppler-utils > /dev/null
!pip install -q -e "."

print("All dependencies installed.")

## 4. Run OCR Pipeline

Output writes to `/kaggle/working/output/` — this is Kaggle's working directory root,
so all `.md` files will be visible in the **Output** tab after **Save Version**.

In [ ]:
import logging
from src.pipeline import run_pipeline

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

run_pipeline(data_dir=DATA_DIR, output_dir=OUTPUT_DIR)

## 5. Check Results

List all generated Markdown files. After verifying, use **Save Version → Save & Run All** to persist outputs.

The files below will appear in your notebook's **Output** tab and can be downloaded or used as a Kaggle Dataset.

In [ ]:
from pathlib import Path

md_files = sorted(Path(OUTPUT_DIR).glob("*.md"))
total_size = sum(f.stat().st_size for f in md_files)

print(f"Generated {len(md_files)} Markdown file(s) ({total_size / 1024:.1f} KB total)\n")
for f in md_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:60s} {size_kb:>8.1f} KB")

print(f"\nOutput location: {OUTPUT_DIR}")
print("These files will be saved in Kaggle's Output tab after Save Version.")

## 6. Preview Output

Preview the first 100 lines of the first generated Markdown file.

In [ ]:
if md_files:
    first_file = md_files[0]
    content = first_file.read_text(encoding="utf-8")
    lines = content.splitlines()
    preview = "\n".join(lines[:100])
    print(f"--- Preview: {first_file.name} ({len(lines)} lines total) ---\n")
    print(preview)
    if len(lines) > 100:
        print(f"\n... ({len(lines) - 100} more lines)")
else:
    print("No output files found.")